# Empirical covariance spectra — years and temperature

Comparison of saved **K=50, threshold=0.1** HDDC EM and KMeans+PCA runs at layers **4 and 28**.
This follows `empirical_covariance_spectra.ipynb`: full training-split empirical covariance spectra,
equal component weights when averaging, then normalization by the largest averaged eigenvalue.

HDDC uses the final checkpoint's **soft posterior memberships**; KMeans uses **hard nearest-centroid memberships**.
Both exports evaluate memberships and recompute covariances around empirical component means in float64, with population (ML) normalization.
KMeans assignments can differ at numerical ties from the float32 training-time assignments.
They are neither fitted model covariance spectra nor spectra of a pooled covariance.
Prompt activations have no synthetic-manifold association or known R/M reference lines, so each dataset/layer is one comparison panel.

Compute once from the repository root:
```bash
sbatch scripts/slurm/temporary/sbatch_sae_empirical_spectra.sh
```
Wait for all eight tasks to complete, then run this notebook. Exports are stored beside each checkpoint;
rerunning the exporter refuses to overwrite existing results.


In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import torch
from IPython.display import display
pio.renderers.default = 'plotly_mimetype'

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "scripts" / "temporary"))
from run_sae_empirical_spectra import selected_runs

N_EIGENVALUES = 20
LOG_SCALE = True
FIGURE_WIDTH = 1600
EXPORT_HTML = True
OUTPUT_DIR = REPO / "outputs/experiments/sae_empirical_covariance_spectra_k50_t0p1"
METHOD_NAMES = {"hddc_em": "HDDC EM (soft)", "kmeans_pca": "KMeans+PCA (hard)"}
COLORS = {"hddc_em": "royalblue", "kmeans_pca": "darkorange"}

runs = selected_runs()
bundles, spectra, summary = {}, {}, []
for run in runs:
    key = (run["dataset"], run["layer"], run["method"])
    bundle = torch.load(run["output"], map_location="cpu", weights_only=True)
    assert bundle["format"] == "dalg_empirical_spectra_v1"
    assert bundle["K"] == 50 and bundle["eigenvalues"].shape == (50, 4096)
    assert Path(bundle["source"]["checkpoint"]).resolve() == run["checkpoint"].resolve()
    assert bundle["source"]["layer"] == run["layer"]
    assert bundle["source"]["split"] == "train"
    assert bundle["computation"]["memberships"] == ("soft" if run["method"] == "hddc_em" else "hard")
    valid = bundle["valid"].numpy()
    values = bundle["eigenvalues"].numpy()
    assert valid.any() and np.isfinite(values[valid]).all() and (values[valid] >= 0).all()
    assert (np.diff(values[valid], axis=1) <= 0).all()
    assert np.isclose(bundle["effective_counts"].sum().item(), bundle["n_activations"])
    bundles[key], spectra[key] = bundle, values
    summary.append({
        "dataset": key[0], "layer": key[1], "method": METHOD_NAMES[key[2]],
        "K": bundle["K"], "threshold": 0.1, "training_rows": bundle["n_activations"],
        "valid_components": int(valid.sum()),
        "min_effective_count": bundle["effective_counts"][bundle["valid"]].min().item(),
        "export": str(run["output"].relative_to(REPO)),
    })

display(pd.DataFrame(summary))
PANELS = list(dict.fromkeys((r["dataset"], r["layer"]) for r in runs))
mean_spectra = {key: values[bundles[key]["valid"].numpy()].mean(axis=0) for key, values in spectra.items()}


,dataset,layer,method,K,threshold,training_rows,valid_components,min_effective_count,export
0,years_neutral,4,HDDC EM (soft),50,0.1,18018,50,91.0,dalg-cache/sae_manifold_years_neutral_llama31_...
1,years_neutral,4,KMeans+PCA (hard),50,0.1,18018,50,34.0,dalg-cache/sae_manifold_years_neutral_llama31_...
2,years_neutral,28,HDDC EM (soft),50,0.1,18018,50,79.0,dalg-cache/sae_manifold_years_neutral_llama31_...
3,years_neutral,28,KMeans+PCA (hard),50,0.1,18018,50,17.0,dalg-cache/sae_manifold_years_neutral_llama31_...
4,temperature_02d,4,HDDC EM (soft),50,0.1,13500,50,25.0,dalg-cache/sae_manifold_temperature_02d_llama3...
5,temperature_02d,4,KMeans+PCA (hard),50,0.1,13500,50,25.0,dalg-cache/sae_manifold_temperature_02d_llama3...
6,temperature_02d,28,HDDC EM (soft),50,0.1,13500,50,20.0,dalg-cache/sae_manifold_temperature_02d_llama3...
7,temperature_02d,28,KMeans+PCA (hard),50,0.1,13500,50,20.0,dalg-cache/sae_manifold_temperature_02d_llama3...


## Average spectra

Each panel overlays HDDC and KMeans for one dataset/layer. Each valid component contributes equally,
regardless of its effective membership count. Display the first 20 eigenvalues by default; the exports retain all 4,096.
All panels share y limits. Exact zeros are omitted on log scales; set `LOG_SCALE = False` to display them.


In [6]:
def plotted(values):
    return np.where(values > 0, values, np.nan) if LOG_SCALE else values


def shared_y_range(curves):
    values = np.concatenate([np.asarray(v).reshape(-1) for v in curves])
    values = values[np.isfinite(values)]
    if LOG_SCALE:
        values = np.log10(values[values > 0])
    assert values.size, "No finite eigenvalues on the selected scale."
    lower, upper = values.min(), values.max()
    padding = 0.05 * (upper - lower) if upper > lower else 0.05 * max(abs(upper), 1.0)
    return [lower - padding, upper + padding]


def finish_figure(fig, filename):
    fig.show()
    if EXPORT_HTML:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.write_html(OUTPUT_DIR / filename, include_plotlyjs=True)


def average_grid(curves, title, y_title):
    displayed = {key: values[:N_EIGENVALUES] for key, values in curves.items()}
    fig = make_subplots(
        rows=2, cols=2, shared_xaxes="all", shared_yaxes="all",
        subplot_titles=[f"{dataset} · layer {layer}" for dataset, layer in PANELS],
        horizontal_spacing=0.08, vertical_spacing=0.16,
    )
    for i, (dataset, layer) in enumerate(PANELS):
        for method in METHOD_NAMES:
            key = (dataset, layer, method)
            values = displayed[key]
            valid_count = int(bundles[key]["valid"].sum())
            fig.add_trace(go.Scatter(
                x=np.arange(1, len(values) + 1), y=plotted(values),
                mode="lines+markers", marker=dict(size=6),
                name=METHOD_NAMES[method], legendgroup=method, showlegend=i == 0,
                line_color=COLORS[method],
                customdata=np.full(len(values), valid_count),
                hovertemplate="Eigenvalue %{x}: %{y:.5g}<br>%{customdata} valid components<extra>%{fullData.name}</extra>",
            ), row=i // 2 + 1, col=i % 2 + 1)
    fig.update_xaxes(title_text="Eigenvalue index (largest first)", dtick=2,
                     range=[0.5, min(N_EIGENVALUES, 4096) + 0.5], showticklabels=True)
    fig.update_yaxes(title_text=y_title, type="log" if LOG_SCALE else "linear",
                     range=shared_y_range(displayed.values()), showticklabels=True)
    fig.update_layout(
        template="plotly_white", title=f"{title} | K=50 | τ=0.1",
        width=FIGURE_WIDTH, height=850, hovermode="x unified",
        legend=dict(orientation="h", y=1.12),
        margin=dict(l=80, r=30, t=140, b=70),
    )
    return fig


finish_figure(average_grid(mean_spectra, "Average empirical spectra", "Eigenvalue"), "average_spectra.html")


## Normalized average spectra

Average across valid components **first**, then divide each run's mean spectrum by its largest eigenvalue.
Each curve starts at one. These panels use their own common y limits.


In [7]:
normalized_mean_spectra = {}
for key, values in mean_spectra.items():
    assert values[0] > 0, f"{key}: normalization requires a positive largest eigenvalue."
    normalized_mean_spectra[key] = values / values[0]

finish_figure(
    average_grid(normalized_mean_spectra, "Normalized average empirical spectra", "λᵢ / λₘₐₓ"),
    "normalized_average_spectra.html",
)


## Individual components

Each panel shows one run. By default, sample five valid component IDs with a fixed seed.
Set `RANDOM_SEED = None` for a fresh sample, or select IDs per run using
`COMPONENT_IDS = {("years_neutral", 4, "hddc_em"): [3, 17]}`.
Component IDs are local to each model; matching IDs across runs do not imply matching regions.


In [8]:
COMPONENT_IDS = {}
N_RANDOM = 20
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)
chosen = {}
for key, bundle in bundles.items():
    valid_ids = np.flatnonzero(bundle["valid"].numpy())
    ids = np.asarray(COMPONENT_IDS[key]) if key in COMPONENT_IDS else rng.choice(
        valid_ids, size=min(N_RANDOM, len(valid_ids)), replace=False,
    )
    assert ids.size and np.isin(ids, valid_ids).all(), f"{key}: select valid component IDs."
    chosen[key] = ids
    print(f"{key}: valid IDs={valid_ids.tolist()}; plotting={ids.tolist()}")

fig = make_subplots(
    rows=4, cols=2, shared_xaxes="all", shared_yaxes="all",
    subplot_titles=[f"{key[0]} · layer {key[1]} · {METHOD_NAMES[key[2]]}" for key in bundles],
    horizontal_spacing=0.08, vertical_spacing=0.08,
)
displayed = []
for i, key in enumerate(bundles):
    for k in chosen[key]:
        values = spectra[key][k, :N_EIGENVALUES]
        displayed.append(values)
        fig.add_trace(go.Scatter(
            x=np.arange(1, len(values) + 1), y=plotted(values), mode="lines+markers",
            name=f"Component {k}", marker=dict(size=5),
        ), row=i // 2 + 1, col=i % 2 + 1)
fig.update_xaxes(title_text="Eigenvalue index", dtick=2,
                 range=[0.5, min(N_EIGENVALUES, 4096) + 0.5], showticklabels=True)
fig.update_yaxes(title_text="Eigenvalue", type="log" if LOG_SCALE else "linear",
                 range=shared_y_range(displayed), showticklabels=True)
fig.update_layout(
    template="plotly_white", title="Individual component spectra | K=50 | τ=0.1",
    width=FIGURE_WIDTH, height=1500, showlegend=False, hovermode="x unified",
    margin=dict(l=80, r=30, t=100, b=70),
)
finish_figure(fig, "individual_component_spectra.html")


('years_neutral', 4, 'hddc_em'): valid IDs=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]; plotting=[24, 32, 7, 41, 48, 33, 6, 2, 21, 46, 15, 3, 22, 30, 39, 31, 36, 14, 26, 40]
('years_neutral', 4, 'kmeans_pca'): valid IDs=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]; plotting=[27, 25, 2, 34, 21, 31, 23, 5, 16, 8, 19, 7, 41, 12, 46, 22, 35, 37, 26, 43]
('years_neutral', 28, 'hddc_em'): valid IDs=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]; plotting=[3, 13, 21, 35, 45, 42, 29, 11, 23, 5, 26, 12, 40, 34, 17, 49, 14, 37, 31, 18]
('years_neutral', 28, 'kmeans_p